# 03 — Cost Analysis

Identifies cost drivers: which DRGs, diagnoses, procedures, and beneficiary profiles are associated with the highest spending.

Requires `cms_data.db` from `01_setup.ipynb`.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

---
## 1. Cost drivers by DRG (Inpatient)

In [3]:
drg_cost = pd.read_sql("""
SELECT
  CLM_DRG_CD as drg,
  COUNT(*) as num_claims,
  ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim,
  ROUND(AVG(CLM_UTLZTN_DAY_CNT), 1) as avg_los
FROM inpatient_claims
WHERE CLM_PMT_AMT > 0
GROUP BY CLM_DRG_CD
ORDER BY total_cost DESC
LIMIT 20
""", conn)

print("Top 20 DRGs by Total Spending")
print("="*90)
display(drg_cost)

Top 20 DRGs by Total Spending


,drg,num_claims,total_cost,avg_cost_per_claim,avg_los
0,939,249,3516800.0,14124.0,11.4
1,948,231,3411700.0,14769.0,10.6
2,949,233,3328000.0,14283.0,11.3
3,945,237,3318000.0,14000.0,11.7
4,951,236,3275000.0,13877.0,10.8
5,946,214,3071100.0,14351.0,11.1
6,941,216,2982600.0,13808.0,10.7
7,950,203,2859900.0,14088.0,11.2
8,864,190,2852000.0,15011.0,7.3
9,862,186,2845530.0,15299.0,8.0


In [4]:
drg_cost.to_csv("../data/exports/drg_cost_analysis.csv", index=False)
print("Saved: data/exports/drg_cost_analysis.csv")

Saved: data/exports/drg_cost_analysis.csv


---
## 2. Cost drivers by primary diagnosis (Inpatient)

In [5]:
diag_cost = pd.read_sql("""
SELECT
  ICD9_DGNS_CD_1 as primary_diagnosis,
  COUNT(*) as num_claims,
  ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim
FROM inpatient_claims
WHERE CLM_PMT_AMT > 0 AND ICD9_DGNS_CD_1 IS NOT NULL
GROUP BY ICD9_DGNS_CD_1
ORDER BY total_cost DESC
LIMIT 20
""", conn)

print("Top 20 Primary Diagnoses by Total Spending (Inpatient)")
print("="*90)
display(diag_cost)
diag_cost.to_csv("../data/exports/diagnosis_cost_analysis.csv", index=False)

Top 20 Primary Diagnoses by Total Spending (Inpatient)


,primary_diagnosis,num_claims,total_cost,avg_cost_per_claim
0,V5789,1772,28421600.0,16039.0
1,0389,1602,24410390.0,15237.0
2,41401,1596,23847800.0,14942.0
3,486,2394,17101860.0,7144.0
4,41071,1124,14809630.0,13176.0
5,51881,795,14232900.0,17903.0
6,71536,1101,13133020.0,11928.0
7,4280,1378,11069900.0,8033.0
8,49121,1520,9999300.0,6578.0
9,5849,1064,8624900.0,8106.0


---
## 3. Impact of chronic conditions on annual spending

In [6]:
chronic_impact = pd.DataFrame()

CONDITION_LABELS = {
    'SP_ALZHDMTA': "Alzheimer's / Dementia",
    'SP_CHF': 'Heart Failure (CHF)',
    'SP_CHRNKIDN': 'Chronic Kidney Disease',
    'SP_CNCR': 'Cancer',
    'SP_COPD': 'COPD',
    'SP_DEPRESSN': 'Depression',
    'SP_DIABETES': 'Diabetes',
    'SP_ISCHMCHT': 'Ischemic Heart Disease',
    'SP_OSTEOPRS': 'Osteoporosis',
    'SP_RA_OA': 'Rheumatoid Arthritis / Osteoarthritis',
    'SP_STRKETIA': 'Stroke / TIA',
}

for condition in ['SP_DIABETES', 'SP_CHF', 'SP_COPD', 'SP_ISCHMCHT']:
    df = pd.read_sql(f"""
    SELECT
      CASE WHEN {condition} = 1 THEN COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) ELSE NULL END as spend_with,
      CASE WHEN {condition} = 2 THEN COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) ELSE NULL END as spend_without
    FROM beneficiary_summary
    """, conn)
    
    row = {
        'condition': CONDITION_LABELS[condition],
        'avg_spend_with_condition': round(df['spend_with'].mean()),
        'avg_spend_without_condition': round(df['spend_without'].mean())
    }
    chronic_impact = pd.concat([chronic_impact, pd.DataFrame([row])], ignore_index=True)

print("Annual Spending Impact: With vs Without Chronic Conditions")
print("="*90)
display(chronic_impact)
chronic_impact.to_csv("../data/exports/chronic_condition_spend_impact.csv", index=False)

Annual Spending Impact: With vs Without Chronic Conditions


,condition,avg_spend_with_condition,avg_spend_without_condition
0,Diabetes,5186,963
1,Heart Failure (CHF),5890,1068
2,COPD,8617,1603
3,Ischemic Heart Disease,4875,745


---
## 4. High-cost beneficiary profiles (top 10%)

In [7]:
spending = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  YEAR,
  COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) + COALESCE(MEDREIMB_CAR, 0) as total_medicare_spend,
  SP_DIABETES,
  SP_CHF,
  SP_COPD,
  SP_ISCHMCHT,
  SP_DEPRESSN,
  SP_ALZHDMTA
FROM beneficiary_summary
""", conn)

p90 = spending['total_medicare_spend'].quantile(0.90)
top_10pct = spending[spending['total_medicare_spend'] >= p90].copy()

print(f"90th percentile spending threshold: ${p90:,.0f}")
print(f"Top 10% of beneficiary-years: {len(top_10pct):,} (out of {len(spending):,})")
print(f"Top 10% avg spend: ${top_10pct['total_medicare_spend'].mean():,.0f}")
print(f"All beneficiaries avg spend: ${spending['total_medicare_spend'].mean():,.0f}")
print(f"\nChronic condition prevalence in top 10%:")
print(f"  Diabetes:           {(top_10pct['SP_DIABETES'] == 1).mean()*100:.1f}%")
print(f"  CHF:                {(top_10pct['SP_CHF'] == 1).mean()*100:.1f}%")
print(f"  COPD:               {(top_10pct['SP_COPD'] == 1).mean()*100:.1f}%")
print(f"  Ischemic Heart Dz:  {(top_10pct['SP_ISCHMCHT'] == 1).mean()*100:.1f}%")
print(f"  Depression:         {(top_10pct['SP_DEPRESSN'] == 1).mean()*100:.1f}%")

90th percentile spending threshold: $8,927
Top 10% of beneficiary-years: 34,365 (out of 343,644)
Top 10% avg spend: $22,356
All beneficiaries avg spend: $3,614

Chronic condition prevalence in top 10%:
  Diabetes:           78.0%
  CHF:                72.1%
  COPD:               45.5%
  Ischemic Heart Dz:  85.0%
  Depression:         47.3%


---
## 5. Spending trends by year

In [8]:
yearly_trends = pd.read_sql("""
SELECT
  YEAR,
  COUNT(DISTINCT DESYNPUF_ID) as num_beneficiaries,
  ROUND(AVG(MEDREIMB_IP), 0) as avg_inpatient_spend,
  ROUND(AVG(MEDREIMB_OP), 0) as avg_outpatient_spend,
  ROUND(AVG(COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) + COALESCE(MEDREIMB_CAR, 0)), 0) as avg_total_spend
FROM beneficiary_summary
GROUP BY YEAR
ORDER BY YEAR
""", conn)

print("Spending Trends: 2008–2010")
print("="*90)
display(yearly_trends)
yearly_trends.to_csv("../data/exports/spending_trends_by_year.csv", index=False)

Spending Trends: 2008–2010


,YEAR,num_beneficiaries,avg_inpatient_spend,avg_outpatient_spend,avg_total_spend
0,2008,116352,2214.0,622.0,3999.0
1,2009,114538,2190.0,770.0,4297.0
2,2010,112754,1242.0,432.0,2522.0


In [9]:
conn.close()
print("Analysis complete. Outputs saved to data/exports/")

Analysis complete. Outputs saved to data/exports/
